# CucuGAN auf Kaggle

**Einstellungen (rechte Seitenleiste → Session options):**
- Accelerator: **GPU T4 x2** (wir nutzen eine GPU)
- Internet: **an** (für `git clone`)
- Input: den Datensatz mit `collage128` hinzufügen ("Add Input")

**Ablauf:** Zelle 1 einstellen → alle Zellen ausführen. Für lange Läufe oben rechts
**Save Version → Save & Run All (Commit)**: läuft dann bis zu 12 h ohne offenen Browser.
Ergebnisse liegen danach im Output der Version (`/kaggle/working/CucuGAN/...`).

**Fortsetzen:** Den Output der vorigen Version als Input hinzufügen ("Add Input" → Notebook Output).
Zelle 3 findet `latest.pt` dann automatisch und trainiert mit `--resume` weiter.

In [ ]:
# 1) Einstellungen: eine der Voreinstellungen aktiv lassen
BRANCH = "gurken-collage"     # welcher Stand des Codes

# Mixed-Precision-Test: 5 Epochen à 200 Schritte, prüft Tempo und ob alles stabil bleibt
RUN, EXTRA = "amp_test", "--max-images 200 --n-epochs 5 --n-epochs-decay 0 --pool-size 0 --amp"

# Overfit-Test: 20 Bilder, 200 Epochen
# RUN, EXTRA = "overfit_gpu", "--max-images 20 --n-epochs 200 --n-epochs-decay 0 --pool-size 0 --save-every 50"

# Baseline: alle Daten, Paper-Standard, mit Mixed Precision
# RUN, EXTRA = "baseline", "--amp"

In [ ]:
# 2) Code holen, GPU und Daten finden
import glob, os, shutil, subprocess
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
if not os.path.exists("/kaggle/working/CucuGAN"):
    !git clone -q -b {BRANCH} https://github.com/Felix-uniulm/CucuGAN.git /kaggle/working/CucuGAN
%cd /kaggle/working/CucuGAN
!git log --oneline -1
!python -c "import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')"

# Datensatz: der Ordner, der train/A enthält
DATA = next(os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/train/A", recursive=True))
print("Daten:", DATA, "|", len(os.listdir(f"{DATA}/train/A")), "Gesichter,", len(os.listdir(f"{DATA}/train/B")), "Gurken")

In [ ]:
# Optional: Geschwindigkeit 32 Bit vs. Mixed Precision (je 1 Epoche à 200 Schritte, ~2 min)
for name, flag in (("speed_fp32", ""), ("speed_amp", "--amp")):
    print(name)
    !python src/train.py --name {name} --data {DATA} --max-images 200 --n-epochs 1 --n-epochs-decay 0 --num-workers 4 {flag} 2>&1 | grep -E "fertig|Error|Traceback"

In [ ]:
# 3) Fortsetzen? Checkpoint aus einem früheren Output übernehmen
prev = glob.glob(f"/kaggle/input/**/checkpoints/{RUN}/latest.pt", recursive=True)
RESUME = ""
if prev:
    os.makedirs(f"checkpoints/{RUN}", exist_ok=True)
    for f in glob.glob(os.path.join(os.path.dirname(prev[0]), "*.pt")):
        shutil.copy(f, f"checkpoints/{RUN}/")
    prev_out = prev[0].replace(f"checkpoints/{RUN}/latest.pt", f"outputs/{RUN}")
    if os.path.exists(prev_out):
        shutil.copytree(prev_out, f"outputs/{RUN}", dirs_exist_ok=True)
    RESUME = "--resume"
    print("Setze fort von", prev[0])
else:
    print("Neuer Lauf")

In [ ]:
# 4) Training
!python src/train.py --name {RUN} --data {DATA} --num-workers 4 {EXTRA} {RESUME} 2>&1 | grep -v "it/s]$"

In [ ]:
# 5) Letztes Kontrollbild anzeigen
from IPython.display import Image, display
samples = sorted(glob.glob(f"outputs/{RUN}/samples/*.png"))
print(len(samples), "Kontrollbilder; zuletzt:", samples[-1] if samples else "-")
if samples:
    display(Image(samples[-1]))